In [13]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

#load .env
load_dotenv()

with open("config/config.json","r") as f:
    config = json.load(f)

print(config)

{'llm': {'openai_model': 'gpt-4o-mini', 'gemini_model': 'gemini-1.5-flash', 'temperature': 0.3}, 'evaluation': {'criteria': ['tone_empathy', 'knowledge_accuracy', 'resolution_quality'], 'score_range': [1, 5]}, 'classification': {'labels': ['billing', 'claims', 'complaint', 'general_query']}}


In [14]:
df = pd.read_csv("data/transcripts.csv")

print("Data Loaded")
print(df.head())

Data Loaded
   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type  
0            billing  
1             claims  
2          complaint  
3      general_query  


In [15]:
df.tail(2)

,call_id,agent_name,transcript,expected_call_type
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [16]:
df.head()

,call_id,agent_name,transcript,expected_call_type
0,1,Alice,Customer: I was charged twice for my premium t...,billing
1,2,Bob,Customer: My claim has been pending for 2 week...,claims
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm(llm):
    provider = os.getenv("LLM_PROVIDER")

    if provider == "openai":
        llm = ChatOpenAI(model = config["llm"].get("openai_model", "gpt-4o-mini"), temperature = config["llm"].get("temperature", 0.3) )
    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(model = config["llm"].get("gemini_model", "gemini-1.5-flash"), temperature = config["llm"].get("temperature", 0.3) )
    else:
        raise ValueError("LLM_PROVIDER")
    return llm

llm = load_llm(config)

print(f"llm initialized using provider: {os.getenv('LLM_PROVIDER')}")

llm initialized using provider: openai


In [18]:
response = llm.invoke("Say 'setup successful' in one short sentence")

print(response)

content='Setup successful.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 16, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_19227531c6', 'id': 'chatcmpl-EVFZsGTVonaMoPiXfKbKHdfuYdExd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a106e0-c383-72b0-a5ce-ac828231054a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 3, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [19]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class ClassificationOutput(BaseModel):
    call_type: str = Field(description = "Type of customer call")
    confidence: float = Field(description = "Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object = ClassificationOutput)


#prompt template
prompt = PromptTemplate(
    template = """
    You are called classification assistant. 

    classify the following customer support transcript into one of the categories:
    {labels}

    Transcript:
    {transcript}

    {format_instructions}   
    """, 
    input_variables = ["transcript"], 
    partial_variables = {
        "format_instructions": parser.get_format_instructions(), 
            "labels": config["classification"]["labels"] 
                  }
)

#chain

classification_chain = prompt | llm | parser

#test chain with one transcript

# df.iloc[0]["transcript"]

response = classification_chain.invoke(df.iloc[0]["transcript"])

print(f"Classification result: {response}")

Classification result: call_type='billing' confidence=0.95


In [20]:
from tqdm import tqdm

results = []

for i , row in tqdm(df.iterrows(), total = len(df), desc = "Classifying Calls"):
    classification_chain = prompt | llm | parser

    try:
        response = classification_chain.invoke(row["transcript"])
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": response.call_type,
            "confidence": response.confidence
        })

    except Exception as e:
        print(f"Error classifying call {row['call_id']}: {e}")
        results.append({
            "call_id": row["call_id"],
            "predicted_call_type": None,
            "confidence": None
        })

results_df = pd.DataFrame(results)
df = df.merge(results_df, on = "call_id")

print(df)

Classifying Calls: 100%|██████████| 4/4 [00:03<00:00,  1.08it/s]

   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type predicted_call_type  confidence  
0            billing             billing        0.95  
1             claims              claims        0.95  
2          complaint           complaint        0.90  
3      general_query       general_query        0.90  


In [21]:
df

,call_id,agent_name,transcript,expected_call_type,predicted_call_type,confidence
0,1,Alice,Customer: I was charged twice for my premium t...,billing,billing,0.95
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,claims,0.95
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,complaint,0.90
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,general_query,0.90


In [22]:
def route_call(call_type):
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution_quality"]
    elif call_type == "complaint":
        return ["tone_empathy", "resolution_quality"]
    elif call_type == "general_query":
        return ["knowledge_accuracy"] 
    else:
        return ["knowledge_accuracy"] 

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("Routing Applied\n")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

Routing Applied

   call_id predicted_call_type                           evaluation_plan
0        1             billing  [knowledge_accuracy, resolution_quality]
1        2              claims  [knowledge_accuracy, resolution_quality]
2        3           complaint        [tone_empathy, resolution_quality]
3        4       general_query                      [knowledge_accuracy]


In [23]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

min_score, max_score = config["evaluation"]["score_range"]


class ToneEvaluationOutput(BaseModel):
    tone_empathy_score: int = Field(description = f"Tone and empathy score between {min_score} and {max_score}", ge = min_score, le = max_score)
    justification: str = Field(description = "Short reason for the score")

tone_parser = PydanticOutputParser(pydantic_object = ToneEvaluationOutput)


#prompt template
tone_prompt = PromptTemplate(
    template = """
    You are a quality assurance assistant for a customer support team.

    Evaluate the agent's tone and empathy in the following customer support transcript.
    Check whether the agent was polite, acknowledged the customer's feelings, and stayed calm and respectful.

    Score it from {min_score} (poor) to {max_score} (excellent).

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": tone_parser.get_format_instructions(),
            "min_score": min_score,
            "max_score": max_score
                  }
)

#chain

tone_chain = tone_prompt | llm | tone_parser

#test chain with one transcript

tone_response = tone_chain.invoke(df.iloc[0]["transcript"])

print(f"Tone evaluation result: {tone_response}")


Tone evaluation result: tone_empathy_score=3 justification="The agent responded promptly but did not acknowledge the customer's feelings about being charged twice, which could have shown more empathy."


In [24]:
from tqdm import tqdm

tone_results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating Tone"):
    # only evaluate calls whose evaluation plan includes tone_empathy
    if "tone_empathy" not in row["evaluation_plan"]:
        continue

    try:
        response = tone_chain.invoke(row["transcript"])
        tone_results.append({
            "call_id": row["call_id"],
            "tone_empathy_score": response.tone_empathy_score,
            "tone_justification": response.justification,
        })

    except Exception as e:
        print(f"Error evaluating tone for call {row['call_id']}: {e}")
        tone_results.append({
            "call_id": row["call_id"],
            "tone_empathy_score": None,
            "tone_justification": None,
        })

tone_df = pd.DataFrame(tone_results, columns=["call_id", "tone_empathy_score", "tone_justification"])
df = df.merge(tone_df, on="call_id", how="left")

print("Tone Evaluation Applied\n")
print(df[["call_id", "predicted_call_type", "tone_empathy_score", "tone_justification"]])


Evaluating Tone: 100%|██████████| 4/4 [00:01<00:00,  3.08it/s]

Tone Evaluation Applied

   call_id predicted_call_type  tone_empathy_score  \
0        1             billing                 NaN   
1        2              claims                 NaN   
2        3           complaint                 3.0   
3        4       general_query                 NaN   

                                  tone_justification  
0                                                NaN  
1                                                NaN  
2  The agent apologized for the inconvenience, wh...  
3                                                NaN  


In [25]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

min_score, max_score = config["evaluation"]["score_range"]


class ResolutionEvaluationOutput(BaseModel):
    resolution_quality_score: int = Field(description = f"Resolution quality score between {min_score} and {max_score}", ge = min_score, le = max_score)
    issue_resolved: bool = Field(description = "True if the customer's issue was fully resolved in the call, otherwise False")
    justification: str = Field(description = "Short reason for the score")

resolution_parser = PydanticOutputParser(pydantic_object = ResolutionEvaluationOutput)


#prompt template
resolution_prompt = PromptTemplate(
    template = """
    You are a quality assurance assistant for a customer support team.

    Evaluate how well the agent resolved the customer's issue in the following customer support transcript.
    Check whether the agent understood the problem, gave a clear solution or next steps, and whether the issue was actually resolved.

    Score it from {min_score} (poor) to {max_score} (excellent).

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": resolution_parser.get_format_instructions(),
            "min_score": min_score,
            "max_score": max_score
                  }
)

#chain

resolution_chain = resolution_prompt | llm | resolution_parser

#test chain with one transcript

resolution_response = resolution_chain.invoke(df.iloc[0]["transcript"])

print(f"Resolution evaluation result: {resolution_response}")


Resolution evaluation result: resolution_quality_score=2 issue_resolved=False justification="The agent acknowledged the customer's issue but did not provide a solution or next steps in the transcript."


In [26]:
from tqdm import tqdm

resolution_results = []

for i, row in tqdm(df.iterrows(), total = len(df), desc = "Evaluating Resolution"):
    # only evaluate calls whose evaluation plan includes resolution_quality
    if "resolution_quality" not in row["evaluation_plan"]:
        continue

    try:
        response = resolution_chain.invoke(row["transcript"])
        resolution_results.append({
            "call_id": row["call_id"],
            "resolution_quality_score": response.resolution_quality_score,
            "issue_resolved": response.issue_resolved,
            "resolution_justification": response.justification
        })

    except Exception as e:
        print(f"Error evaluating resolution for call {row['call_id']}: {e}")
        resolution_results.append({
            "call_id": row["call_id"],
            "resolution_quality_score": None,
            "issue_resolved": None,
            "resolution_justification": None
        })

resolution_df = pd.DataFrame(resolution_results, columns = ["call_id", "resolution_quality_score", "issue_resolved", "resolution_justification"])
df = df.merge(resolution_df, on = "call_id", how = "left")

print("Resolution Evaluation Applied\n")
print(df[["call_id", "predicted_call_type", "resolution_quality_score", "issue_resolved"]])


Evaluating Resolution: 100%|██████████| 4/4 [00:07<00:00,  1.85s/it]

Resolution Evaluation Applied

   call_id predicted_call_type  resolution_quality_score issue_resolved
0        1             billing                       2.0          False
1        2              claims                       2.0          False
2        3           complaint                       2.0          False
3        4       general_query                       NaN            NaN


In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

min_score, max_score = config["evaluation"]["score_range"]


class KnowledgeEvaluationOutput(BaseModel):
    knowledge_accuracy_score: int = Field(description = f"Knowledge accuracy score between {min_score} and {max_score}", ge = min_score, le = max_score)
    justification: str = Field(description = "Short reason for the score")

knowledge_parser = PydanticOutputParser(pydantic_object = KnowledgeEvaluationOutput)


#prompt template
knowledge_prompt = PromptTemplate(
    template = """
    You are a quality assurance assistant for an insurance customer support team.

    Evaluate the accuracy of the information the agent gave in the following customer support transcript.
    Check whether the agent's answers were correct, clear and relevant to the customer's question,
    and whether the agent avoided guessing or giving misleading information.

    Score it from {min_score} (poor) to {max_score} (excellent).

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": knowledge_parser.get_format_instructions(),
            "min_score": min_score,
            "max_score": max_score
                  }
)

#chain

knowledge_chain = knowledge_prompt | llm | knowledge_parser

#test chain with one transcript

knowledge_response = knowledge_chain.invoke(df.iloc[0]["transcript"])

print(f"Knowledge evaluation result: {knowledge_response}")


Knowledge evaluation result: knowledge_accuracy_score=3 justification="The agent's initial response to check the customer's concern about being charged twice is appropriate, but there is no information provided in the transcript about the resolution or explanation of the issue. Therefore, while the agent did not provide incorrect information, the lack of clarity and resolution leads to a moderate score."


In [28]:
from tqdm import tqdm

knowledge_results = []

for i, row in tqdm(df.iterrows(), total = len(df), desc = "Evaluating Knowledge"):
    # only evaluate calls whose evaluation plan includes knowledge_accuracy
    if "knowledge_accuracy" not in row["evaluation_plan"]:
        continue

    try:
        response = knowledge_chain.invoke(row["transcript"])
        knowledge_results.append({
            "call_id": row["call_id"],
            "knowledge_accuracy_score": response.knowledge_accuracy_score,
            "knowledge_justification": response.justification
        })

    except Exception as e:
        print(f"Error evaluating knowledge for call {row['call_id']}: {e}")
        knowledge_results.append({
            "call_id": row["call_id"],
            "knowledge_accuracy_score": None,
            "knowledge_justification": None
        })

knowledge_df = pd.DataFrame(knowledge_results, columns = ["call_id", "knowledge_accuracy_score", "knowledge_justification"])
df = df.merge(knowledge_df, on = "call_id", how = "left")

print("Knowledge Evaluation Applied\n")
print(df[["call_id", "predicted_call_type", "knowledge_accuracy_score"]])


Evaluating Knowledge: 100%|██████████| 4/4 [00:05<00:00,  1.31s/it]

Knowledge Evaluation Applied

   call_id predicted_call_type  knowledge_accuracy_score
0        1             billing                       3.0
1        2              claims                       2.0
2        3           complaint                       NaN
3        4       general_query                       3.0
